# Treinamento com interface de alto nível

## Importação das bibliotecas

In [79]:
# http://pytorch.org/
from os.path import exists

import torch

In [80]:
import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.transforms import v2
from torch.utils.data import Dataset, random_split
from torch.optim.lr_scheduler import StepLR
from PIL import UnidentifiedImageError

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split

# Dataset

In [3]:
#!/bin/bash
!curl -L -o /content/microsoft-catsvsdogs-dataset.zip https://www.kaggle.com/api/v1/datasets/download/shaunthesheep/microsoft-catsvsdogs-dataset

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  787M  100  787M    0     0   172M      0  0:00:04  0:00:04 --:--:--  195M


In [7]:
!unzip -q -o /content/microsoft-catsvsdogs-dataset.zip -d /content/icrosoft-catsvsdogs-dataset

In [81]:
cats_dogs_folder = "/content/icrosoft-catsvsdogs-dataset/PetImages"

In [82]:
transform = transforms.Compose([
    transforms.Resize((100,100)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
    ])

In [83]:
class CatsDogsDataset(datasets.ImageFolder):
  def __getitem__(self, index):
    try:
      return super(CatsDogsDataset, self).__getitem__(index)
    except UnidentifiedImageError:
      print(f"Pulou Imagem problematica no indice {index}")
      return None

In [84]:
dataset = CatsDogsDataset(cats_dogs_folder, transform)

In [85]:
dataset_train, dataset_val = random_split(dataset, [0.7, 0.3])

In [86]:
len(dataset_train)

17500

In [87]:
len(dataset_val)

7500

## Criação da rede

In [88]:
input_size = 100*100
output_size = 2
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_size, 2048),
            nn.ReLU(),
            nn.Linear(2048, 4098),
            nn.ReLU(),
            nn.Linear(4098, 8192),
            nn.ReLU(),
            nn.Linear(8192, 4098),
            nn.ReLU(),
            nn.Linear(4098, 2048),
            nn.ReLU(),
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, output_size)
        )

    def forward(self, x):
        x = x.view(-1, input_size)
        x = self.fc(x)
        output = F.log_softmax(x, dim=1)
        return output

model = Net()

In [89]:
model

Net(
  (fc): Sequential(
    (0): Linear(in_features=10000, out_features=2048, bias=True)
    (1): ReLU()
    (2): Linear(in_features=2048, out_features=4098, bias=True)
    (3): ReLU()
    (4): Linear(in_features=4098, out_features=8192, bias=True)
    (5): ReLU()
    (6): Linear(in_features=8192, out_features=4098, bias=True)
    (7): ReLU()
    (8): Linear(in_features=4098, out_features=2048, bias=True)
    (9): ReLU()
    (10): Linear(in_features=2048, out_features=1024, bias=True)
    (11): ReLU()
    (12): Linear(in_features=1024, out_features=512, bias=True)
    (13): ReLU()
    (14): Linear(in_features=512, out_features=256, bias=True)
    (15): ReLU()
    (16): Linear(in_features=256, out_features=2, bias=True)
  )
)

## Treinamento

In [90]:
model = Net()

In [91]:
model(dataset_train[10][0])

tensor([[-0.6889, -0.6974]], grad_fn=<LogSoftmaxBackward0>)

### Criando o objeto de treinamento

In [92]:
def train(log_interval, dry_run, model, device, train_loader, optimizer, epoch, criterion):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
            if dry_run:
                break

In [93]:
def test(model, device, test_loader, criterion):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    acc = 100. * correct / len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        acc))
    return acc

## Avaliação

In [94]:
def custom_collate(batch):
  batch = list(filter(lambda x: x is not None, batch))
  return torch.utils.data.default_collate(batch)

In [95]:
use_cuda = torch.cuda.is_available()

torch.manual_seed(1111)

device = torch.device("cuda" if use_cuda else "cpu")

train_kwargs = {'batch_size': 9000}
test_kwargs = {'batch_size': 2000}
if use_cuda:
    cuda_kwargs = {'num_workers': 1,
                    'pin_memory': True,
                    'shuffle': True}
    train_kwargs.update(cuda_kwargs)
    test_kwargs.update(cuda_kwargs)

train_loader = torch.utils.data.DataLoader(dataset_train,**train_kwargs, collate_fn=custom_collate)
test_loader = torch.utils.data.DataLoader(dataset_val, **test_kwargs, collate_fn=custom_collate)
model = Net().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = F.nll_loss
best_acc = test(model, device, test_loader, criterion)
epochs = 8
scheduler = StepLR(optimizer, step_size=4, gamma=0.7)

for epoch in range(1, epochs + 1):
    train(10, False, model, device, train_loader, optimizer, epoch, criterion)
    acc = test(model, device, test_loader, criterion)
    if (acc < best_acc):
      best_acc = acc
      torch.save(model.state_dict(), "cats_dogs_nn.pt")
    scheduler.step()

Pulou Imagem problematica no indice 14395

Test set: Average loss: 0.6931, Accuracy: 3762/7500 (50%)



/usr/local/lib/python3.11/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Train Epoch: 1 [0/17500 (0%)]	Loss: 0.693237
Pulou Imagem problematica no indice 8790
Pulou Imagem problematica no indice 14395

Test set: Average loss: 0.6941, Accuracy: 3320/7500 (44%)



/usr/local/lib/python3.11/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Train Epoch: 2 [0/17500 (0%)]	Loss: 0.694271
Pulou Imagem problematica no indice 8790
Pulou Imagem problematica no indice 14395

Test set: Average loss: 0.7826, Accuracy: 3737/7500 (50%)

Pulou Imagem problematica no indice 8790
Train Epoch: 3 [0/17500 (0%)]	Loss: 0.782196


...

Pulou Imagem problematica no indice 14395

Test set: Average loss: 0.6928, Accuracy: 3762/7500 (50%)



/usr/local/lib/python3.11/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Train Epoch: 7 [0/17500 (0%)]	Loss: 0.693073
Pulou Imagem problematica no indice 8790
Pulou Imagem problematica no indice 14395

Test set: Average loss: 0.6923, Accuracy: 3890/7500 (52%)



/usr/local/lib/python3.11/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Pulou Imagem problematica no indice 8790
Train Epoch: 8 [0/17500 (0%)]	Loss: 0.692295
Pulou Imagem problematica no indice 14395

Test set: Average loss: 0.6913, Accuracy: 4108/7500 (55%)

